# 03 - Train, validate and export the affinity model 📈

This notebook downloads the fixed embedding dataset, trains the regression head, reports held-out metrics, exports ONNX, runs a parity check, and uploads the final model. The default dataset contains ESM-2 and MoLFormer embeddings; the legacy profile remains selectable.

In [1]:
WK_DIR = "/content/"

In [2]:
%cd {WK_DIR}
!test -d Protein-Ligand-Affinity-Prediction-Using-LLM || git clone https://github.com/karthikeyanr103/Protein-Ligand-Affinity-Prediction-Using-LLM.git
%cd {WK_DIR}/Protein-Ligand-Affinity-Prediction-Using-LLM
%pip install -e ".[runtime]" "huggingface-hub>=0.30" onnx

/content
/content/Protein-Ligand-Affinity-Prediction-Using-LLM
Obtaining file:///content/Protein-Ligand-Affinity-Prediction-Using-LLM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for protein-compound-affinity (pyproject.toml) ... done
  Created wheel for protein-compound-affinity: filename=protein_compound_affinity-0.1.0-0.editable-py3-none-any.whl size=6087 sha256=cef759053e5085169b37b2c2c9cc0f73b20f6e1097337a26e1fda34f208a759d
  Stored in directory: /tmp/pip-ephem-wheel-cache-st3gy0sg/wheels/22/c2/79/b5cee1faaf24add23b4f22a8d21051d47ca93d9993ad87cecd
Successfully built protein-compound-affinity
  Attempting uninstall: protein-compound-affinity
    Found existing installation: protein-compound-affinity 0.1.0
    Uninstalling protein-compound-affinity-0.1.0:
      Successfully uninstalled protein-compound-

In [3]:
from google.colab import userdata
from huggingface_hub import HfApi, login, snapshot_download
from pathlib import Path
import os, subprocess, shutil

HF_USER = userdata.get('HF_USER')
ENCODER_PROFILE = 'lightweight'  # lightweight or legacy
DATASET_REPO = f'{HF_USER}/protein-compound-affinity-esm2-molformer'
MODEL_REPO = f'{HF_USER}/protein-compound-affinity-esm2-molformer-onnx'
if ENCODER_PROFILE == 'legacy':
    DATASET_REPO = f'{HF_USER}/protein-compound-affinity-embeddings'
    MODEL_REPO = f'{HF_USER}/protein-compound-affinity-onnx'
ROOT = Path(f'{WK_DIR}/protein_affinity')
OUTPUT = ROOT / 'affinity_model'
OUTPUT.mkdir(parents=True, exist_ok=True)
token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = token
login(token=token)
DATASET = Path(snapshot_download(DATASET_REPO, repo_type='dataset', token=token))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

## Training configuration

In [4]:
config = ROOT / 'training.toml'
config.write_text(f'''[data]
path = "{DATASET}"
seed = 42

[model]
hidden_dims = [1024, 512, 128]
dropout = 0.25

[training]
batch_size = 512
epochs = 100
learning_rate = 0.0005
weight_decay = 0.0001
patience = 6
num_workers = 2

[output]
directory = "{OUTPUT}"
''')
print(config.read_text())

[data]
path = "/root/.cache/huggingface/hub/datasets--IAmKarthik--protein-compound-affinity-esm2-molformer/snapshots/7918e8ba7e37e8558ddf47e3232b37fff22fee68"
seed = 42

[model]
hidden_dims = [1024, 512, 128]
dropout = 0.25

[training]
batch_size = 512
epochs = 100
learning_rate = 0.0005
weight_decay = 0.0001
patience = 6
num_workers = 2

[output]
directory = "/content/protein_affinity/affinity_model"



## Train and evaluate

In [5]:
!affinity-train --config {str(config)}

{"epoch": 1, "train_mse": 2.623457155547868, "validation_mae": 1.0478456808550416, "validation_rmse": 1.310989229109481, "validation_r2": 0.2048315065308981, "validation_pearson": 0.5050363854287914}
{"epoch": 2, "train_mse": 1.4642785325734817, "validation_mae": 1.059931663734609, "validation_rmse": 1.327489150115413, "validation_r2": 0.18468979733110646, "validation_pearson": 0.49391376310777596}
{"epoch": 3, "train_mse": 1.2903314826813148, "validation_mae": 1.0303378049407612, "validation_rmse": 1.291313004774291, "validation_r2": 0.228521253208274, "validation_pearson": 0.5298017603158804}
{"epoch": 4, "train_mse": 1.1885718752898526, "validation_mae": 1.0384654933889252, "validation_rmse": 1.2984498664959216, "validation_r2": 0.21997003085758315, "validation_pearson": 0.5342709767287113}
{"epoch": 5, "train_mse": 1.109724865743375, "validation_mae": 1.046710984696793, "validation_rmse": 1.3144237038856896, "validation_r2": 0.20065975082980103, "validation_pearson": 0.524564352911

In [6]:
!affinity-evaluate --artifacts {str(OUTPUT)}

{
  "mae": 0.9924523358160282,
  "rmse": 1.2523850900494435,
  "r2": 0.20721076783705172,
  "pearson": 0.4808181420395148
}


In [7]:
import json, pandas as pd
metadata = json.loads((OUTPUT / 'metadata.json').read_text())
history = pd.read_json(OUTPUT / 'history.json')
print('Test metrics:', metadata['test_metrics'])
history.tail()

Test metrics: {'mae': 0.99245233581881, 'rmse': 1.252385090326796, 'r2': 0.2072107675238175, 'pearson': 0.4808181418769586}


,epoch,train_mse,validation_mae,validation_rmse,validation_r2,validation_pearson
4,5,1.109725,1.046711,1.314424,0.200660,0.524564
5,6,1.045358,1.027365,1.296399,0.222432,0.540585
6,7,0.994064,1.049048,1.308300,0.208090,0.510233
7,8,0.944943,1.035458,1.299355,0.218882,0.520982
8,9,0.900592,1.050394,1.322170,0.191210,0.525903


## Export the trained head to ONNX

In [11]:
!affinity-export-onnx --artifacts {str(OUTPUT)}
shutil.copy2('MODEL_CARD.md', OUTPUT / 'README.md')
print(sorted(path.name for path in OUTPUT.iterdir()))

/content/protein_affinity/affinity_model/model.onnx
['README.md', 'history.json', 'metadata.json', 'model.onnx', 'model.pt', 'normalization.npz', 'test_predictions.csv']


## Upload the trained model

In [12]:
UPLOAD = True
if UPLOAD:
    api = HfApi(token=token)
    api.create_repo(MODEL_REPO, repo_type='model', exist_ok=True)
    api.upload_folder(
        repo_id=MODEL_REPO,
        repo_type='model',
        folder_path=str(OUTPUT),
        allow_patterns=['model.onnx', 'normalization.npz', 'metadata.json', 'README.md'],
    )
    print('Uploaded:', MODEL_REPO)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...y_model/normalization.npz:   7%|7         |   709B / 9.53kB            

  ...affinity_model/model.onnx:   7%|7         |  559kB / 7.50MB            

Uploaded: IAmKarthik/protein-compound-affinity-esm2-molformer-onnx
